In this example, we will use the IEEEPPG dataset @{tan2020}, which focuses on heart rate monitoring during physical exercise using wrist-type _photoplethysmographic_ (PPG) signals.
The dataset consists of two PPG signals and three-axis acceleration signals.
The goal is to predict ECG values from the PPG and acceleration signals.
All signals were sampled at 125 Hz.

### Imports

In [ ]:
# | code-fold: true

import tempfile

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import requests
from sklearn.metrics import root_mean_squared_error
from sklearn.preprocessing import MinMaxScaler
from sktime.datasets import load_from_tsfile_to_dataframe
from sktime.regression.deep_learning import CNNRegressor
from tensorflow.keras.optimizers import Adam
from tslearn.preprocessing import TimeSeriesScalerMinMax

np.random.seed(42)  # for reproducibility

### Loading the dataset

In [ ]:
# | code-fold: true


def download_data(url: str) -> tuple:
    """Download and load a time series dataset from a URL.

    Args:
        url: URL to the .ts file.

    Returns:
        Tuple of (X, y) DataFrames/Series.
    """
    with requests.get(url, stream=True) as r:
        r.raise_for_status()

        with tempfile.NamedTemporaryFile(suffix=".ts", delete=False) as f:
            for chunk in r.iter_content(chunk_size=8192):
                f.write(chunk)

            fp = f.name

    X, y = load_from_tsfile_to_dataframe(fp)

    return X, y


testdata_url = "https://zenodo.org/records/3902710/files/IEEEPPG_TEST.ts?download=1"
traindata_url = "https://zenodo.org/records/3902710/files/IEEEPPG_TRAIN.ts?download=1"

X_test_raw, y_test_raw = download_data(testdata_url)
X_train_raw, y_train_raw = download_data(traindata_url)

y_train_raw = y_train_raw.astype(float)
y_test_raw = y_test_raw.astype(float)

First we look at the raw training data.
Each row corresponds to one sample in the dataset.
The data comes in five columns, where each column represents a different signal source (PPG or acceleration signal).
Note that each entry in the DataFrame is itself a time series.

In [ ]:
X_train_raw

In [ ]:
y_train_raw

To get a clearer picture, we can plot the signals of some of the training samples.

In [ ]:
sampled = X_train_raw.sample(3, random_state=1337).sort_index()

fig, axes = plt.subplots(
    nrows=sampled.shape[1], ncols=3, figsize=(16, 3 * sampled.shape[1]), sharex=True
)

for row_idx, col in enumerate(sampled.columns):
    for col_idx, (i, row) in enumerate(sampled.iterrows()):
        axes[row_idx, col_idx].plot(row[col].values, label=f"Sample {i}", linewidth=0.8)
        axes[row_idx, col_idx].set_title(f"{col} - Heart rate {y_train_raw[i]:.0f}")
        axes[row_idx, col_idx].set_xlabel("Time")
    axes[row_idx, 0].set_ylabel("Value")

plt.tight_layout()
plt.show()

The following plot shows the distribution of the target values in the training set.

In [ ]:
plt.figure(figsize=(6, 4))
plt.hist(y_train_raw, bins=30, alpha=0.7)
plt.xlabel("Target Value")
plt.ylabel("Frequency")
plt.title("Histogram of y_train")
plt.show()

To feed the data into the CNN, we first need to convert the dataframe into a so-called tensor (in this case a 3D array).

In [ ]:
def convert_df_to_tensor(df: pd.DataFrame) -> np.ndarray:
    """Convert a DataFrame of time series to a 3D numpy tensor.

    Args:
        df: DataFrame where each cell contains a time series.

    Returns:
        3D array of shape (n_samples, ts_length, n_features).
    """
    tensor = np.array(
        [[df.iloc[i, j].values for j in range(df.shape[1])] for i in range(len(df))]
    )
    tensor = np.transpose(tensor, (0, 2, 1))

    for i, j in zip(range(len(df)), range(len(df.columns))):
        assert np.array_equal(df.iloc[i, j].values, tensor[i, :, j])

    return tensor


X_train_tensor = convert_df_to_tensor(X_train_raw)
X_test_tensor = convert_df_to_tensor(X_test_raw)

Let us check the shapes orf the resulting tensors.

In [ ]:
print(X_train_tensor.shape)
print(X_test_tensor.shape)

The training data set has 1768 samples, each consisting of 5 channels (PPG and acceleration signals) with a length of 1000 time steps (which are 8 seconds at 125 Hz).

Neural networks are generally trained on normalized data.
We will use a leaky ReLU activation function in our CNN, which works best with input data within a [0, 1] range.

Unfortunately the `TimeSeriesScalerMinMax` from `tslearn` needs the data in the shape (n_samples, n_timestamps, n_channels), while the CNN expects (n_samples, n_channels, n_timestamps), so we need to swap the last two axes again after scaling.

In [ ]:
scaler_x = TimeSeriesScalerMinMax()
X_train = scaler_x.fit_transform(X_train_tensor, per_feature=True)
X_test = scaler_x.transform(X_test_tensor, per_feature=True)

scaler_y = MinMaxScaler()
y_train = scaler_y.fit_transform(y_train_raw.reshape(-1, 1))
y_test = scaler_y.transform(y_test_raw.reshape(-1, 1))

X_train = np.transpose(X_train, (0, 2, 1))
X_test = np.transpose(X_test, (0, 2, 1))

In [ ]:
print(X_train.shape)
print(X_test.shape)

### Regressor

The `CNNRegressor` from `sktime` can now be used to create and train the convolutional neural network for time series regression.

In [ ]:
regressor = CNNRegressor(
    activation="linear",
    activation_hidden="leaky_relu",
    batch_size=64,
    metrics=[],
    n_conv_layers=5,
    n_epochs=80,
    optimizer=Adam(),
    random_state=42,
    verbose=True,
)

regressor.fit(X_train, y_train)

To verify the performance of the trained model, we now predict the target values for the test set and compute the _root mean squared error_ (RMSE) between the predicted and true target values.

Keep in mind, that we have to inverse transform the predicted target values back to the original scale before computing the RMSE.

In [ ]:
y_pred_scaled = regressor.predict(X_test)
y_pred = scaler_y.inverse_transform(y_pred_scaled.reshape(-1, 1))

rmse = root_mean_squared_error(y_test_raw, y_pred)
print(f"RMSE: {rmse:.4f}")

The RMSE indicates that the model is not performing very well on this task.

Let us visualize the residuals (the difference between the true and predicted target values) to get a better understanding of the model's performance.

In [ ]:
plt.figure(figsize=(6, 4))
plt.hist(y_test_raw - y_pred.flatten(), bins=30, alpha=0.7)
plt.xlabel("Residual (True - Predicted)")
plt.ylabel("Frequency")
plt.title("Histogram of Prediction Residuals")
plt.show()

A scatter plot showing true against predicted values reveals that the model tends to overestimate smaller values and underestimate larger values.

In [ ]:
plt.figure(figsize=(6, 6))
plt.hexbin(y_test_raw, y_pred.flatten(), gridsize=40, cmap="viridis", mincnt=1)
plt.xlabel("True Values")
plt.ylabel("Predicted Values")
plt.title("Hexbin Density Plot of True vs Predicted Values")
plt.plot(
    [y_test_raw.min(), y_test_raw.max()], [y_test_raw.min(), y_test_raw.max()], "r--"
)
plt.colorbar(label="Counts")
plt.show()

Especially when encountering a model that performs worse than expected, it is important to analyze potential reasons for this behavior.

Here, we predict the training set using the trained model.
Usually, the training error should be significantly lower than the test error.
If this is not the case, we can suspect issues with the code, model (architecture) or with the data itself.

_Plotting the histogram of residuals on training set_

In [ ]:
y_pred_train_scaled = regressor.predict(X_train)
y_pred_train = scaler_y.inverse_transform(y_pred_train_scaled.reshape(-1, 1))

plt.figure(figsize=(6, 4))
plt.hist(y_train_raw - y_pred_train.flatten(), bins=30, alpha=0.7)
plt.xlabel("Residual (True - Predicted)")
plt.ylabel("Frequency")
plt.title("Histogram of Prediction Residuals")
plt.show()

_Plotting the scatter plot of true vs. predicted values on training set_

In [ ]:
plt.figure(figsize=(6, 6))
plt.hexbin(y_train_raw, y_pred_train.flatten(), gridsize=40, cmap="viridis", mincnt=1)
plt.xlabel("True Values")
plt.ylabel("Predicted Values")
plt.title("Hexbin Density Plot of True vs Predicted Values (Train)")
plt.plot(
    [y_train_raw.min(), y_train_raw.max()],
    [y_train_raw.min(), y_train_raw.max()],
    "r--",
)
plt.colorbar(label="Counts")
plt.show()

The model is able to fit the training data quite well, indicating that the code is implemented correctly and the model architecture is complex enough to capture the underlying patterns in the data.
Still, the performance on the test set is significantly worse than on the training set.
This suggests that the model is likely overfitting the data, which could be due to various factors such as e.g. insufficient training data, or lack of regularization or that the test data distribution differs significantly from the training data distribution.

While these results are not fully satisfying, the results align with those reported by @{tan2021} for other simpler neural network architectures on this dataset.